# Multi-Model Agent Patterns: Runtime Provider Switching

This notebook demonstrates how Strands Agents decouple model configuration from application logic. You'll learn to:

- Create an agent with a specific model provider
- Swap the model provider at runtime without modifying tools or prompts
- Verify that tools and system prompts persist across provider switches

**Key Insight**: Because Strands separates the *agent logic* (tools, prompts, conversation) from the *model backend*, you can switch providers dynamically — useful for A/B testing, cost optimization, or graceful degradation.

## Environment Setup

First, we validate that the required environment is configured. This notebook uses Amazon Bedrock, so you need valid AWS credentials (via environment variables or AWS CLI configuration).

> **Note**: On SageMaker Studio, credentials are provided automatically via the IAM execution role. You may see warnings about missing `AWS_ACCESS_KEY_ID` — these are optional and can be safely ignored in that environment.

In [ ]:
import os


def check_environment(required_vars: list[str], optional_vars: list[str]) -> None:
    """Validate environment variables are set for model providers.

    Args:
        required_vars: Environment variables that must be set.
        optional_vars: Environment variables that enhance the tutorial but aren't required.
    """
    missing_required = [v for v in required_vars if not os.environ.get(v)]
    missing_optional = [v for v in optional_vars if not os.environ.get(v)]

    if missing_required:
        print("❌ Missing REQUIRED environment variables:")
        for var in missing_required:
            print(f"   - {var}")
        print("\nSet these before running the notebook.")
        raise EnvironmentError(f"Missing required variables: {missing_required}")

    if missing_optional:
        print("⚠️  Missing OPTIONAL environment variables (some examples will use alternatives):")
        for var in missing_optional:
            print(f"   - {var}")
    else:
        print("✅ All environment variables are set.")


# Bedrock uses AWS credentials from the environment or ~/.aws/credentials
# These are the standard AWS credential variables
check_environment(
    required_vars=["AWS_DEFAULT_REGION"],
    optional_vars=["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]
)

## Imports and Error Handling

We import the Strands SDK components and define a `safe_agent_call()` wrapper that provides informative error messages when a provider fails. This pattern is essential for production use — it catches common issues like missing credentials, throttling, and timeouts, and suggests corrective actions.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands import tool


def safe_agent_call(agent: Agent, task: str, provider_name: str) -> str:
    """Execute an agent call with informative error handling.

    Wraps agent invocation to catch provider errors and display actionable
    diagnostics instead of raw stack traces.

    Args:
        agent: The Strands Agent instance to invoke.
        task: The task/prompt to send to the agent.
        provider_name: Human-readable name of the provider for error messages.

    Returns:
        The agent's response as a string, or empty string on failure.
    """
    try:
        response = agent(task)
        return str(response)
    except Exception as e:
        error_type = type(e).__name__
        print(f"❌ Error from {provider_name}: {error_type}")
        print(f"   Message: {str(e)}")

        # Suggest corrective action based on error type
        if "credential" in str(e).lower() or "key" in str(e).lower():
            print(f"   → Check that API keys for {provider_name} are set in environment variables")
        elif "throttl" in str(e).lower() or "rate" in str(e).lower():
            print(f"   → {provider_name} is rate-limited. Wait and retry, or use a different provider")
        elif "timeout" in str(e).lower():
            print(f"   → {provider_name} timed out. The model may be overloaded")
        else:
            print(f"   → Verify model ID and region configuration for {provider_name}")

        return ""

## Define a Reusable Tool

We define a simple tool using the `@tool` decorator. This tool performs an observable action (counting words) so we can verify it works after a model swap. The key point: **tools persist across provider switches**.

In [ ]:
@tool
def word_count(text: str) -> str:
    """Count the number of words in the given text.

    Args:
        text: The text content to count words in.

    Returns:
        A string reporting the word count.
    """
    count = len(text.split())
    return f"The text contains {count} words."

## Create an Agent with Claude Sonnet

We create our first model instance using Amazon Bedrock's Claude Sonnet — a highly capable model suitable for complex reasoning tasks. The agent is configured with:
- A specific model provider (`sonnet_model`)
- A tool (`word_count`)
- A system prompt that persists across provider switches

In [ ]:
# Create a Claude Sonnet model instance via Bedrock
sonnet_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name="us-east-1"
)

# Create the agent with a system prompt and tool
agent = Agent(
    model=sonnet_model,
    tools=[word_count],
    system_prompt="You are a helpful assistant that provides clear, concise answers. Always identify yourself by stating which task you are performing.",
    callback_handler=None, load_tools_from_directory=False
)

print("✅ Agent created with Claude Sonnet model and word_count tool")
print(f"   Model: us.anthropic.claude-sonnet-4-5-20250929-v1:0")
print(f"   Tools: {[t.__name__ for t in [word_count]]}")

## Use the Agent with Claude Sonnet

Let's invoke the agent with a task. The response comes from Claude Sonnet via Bedrock.

In [ ]:
# Invoke the agent with Claude Sonnet as the backend
response_sonnet = safe_agent_call(
    agent,
    "What are three benefits of using multiple AI models in a single application? Keep it brief.",
    provider_name="Claude Sonnet (Bedrock)"
)

print(f"\n[Claude Sonnet Response]\n{response_sonnet}")

## Runtime Provider Switch: Swap to Nova Lite

Here's the key pattern: **we swap the model at runtime by reassigning `agent.model`**. The agent keeps its tools, system prompt, and conversation history — only the underlying model changes.

Nova Lite is a faster, more cost-effective model suitable for simpler tasks. By swapping at runtime, you can route different tasks to different models without creating new agents.

In [ ]:
# Create a Nova Lite model instance — faster and cheaper than Sonnet
nova_model = BedrockModel(
    model_id="us.amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

# Runtime provider switch: swap the model without touching tools or prompts
agent.model = nova_model

print("✅ Model swapped to Nova Lite at runtime")
print("   Tools and system prompt remain unchanged.")

## Verify the Swap: Same Task, Different Model

We send the same task to the agent, now backed by Nova Lite. The response style may differ (each model has its own characteristics), but the agent's behavior — including tool access and system prompt adherence — remains consistent.

In [ ]:
# Same agent, same task — but now using Nova Lite
response_nova = safe_agent_call(
    agent,
    "What are three benefits of using multiple AI models in a single application? Keep it brief.",
    provider_name="Nova Lite (Bedrock)"
)

print(f"\n[Nova Lite Response]\n{response_nova}")

## Verify Tools Persist After Swap

A critical property of runtime provider switching: **tools remain functional after the model swap**. The agent can still invoke `word_count` even though the underlying model changed from Claude Sonnet to Nova Lite.

In [ ]:
# Verify the word_count tool still works after the model swap
sample_article = (
    "Artificial intelligence is transforming healthcare by enabling faster diagnosis, "
    "personalized treatment plans, and drug discovery. Machine learning models can analyze "
    "medical images with accuracy comparable to specialists, while natural language processing "
    "helps extract insights from clinical notes."
)

response_tool_test = safe_agent_call(
    agent,
    f"Use the word_count tool on this text: '{sample_article}'",
    provider_name="Nova Lite (Bedrock)"
)

print(f"\n[Nova Lite + Tool] {response_tool_test}")
print("\n✅ Tool invocation successful after provider switch!")

## Verify System Prompt Persists After Swap

The system prompt we set during agent creation ("You are a helpful assistant...") should still guide the model's behavior after the swap. Let's verify by asking the agent to follow a directive from its system prompt.

In [ ]:
# The system prompt instructed the agent to "identify yourself by stating which task you are performing"
# This should still be in effect after the model swap
response_system_prompt_test = safe_agent_call(
    agent,
    "Explain what runtime provider switching means in one sentence.",
    provider_name="Nova Lite (Bedrock)"
)

print(f"\n[Nova Lite - System Prompt Test]\n{response_system_prompt_test}")
print("\n✅ System prompt persists after provider switch!")

## Summary

In this notebook, you learned:

1. **Agent creation**: Create an agent with a specific `BedrockModel` instance, tools, and system prompt
2. **Runtime switching**: Swap the model via `agent.model = new_model` — no need to recreate the agent
3. **Tool persistence**: Tools (like `word_count`) remain fully functional after a provider switch
4. **System prompt persistence**: The system prompt continues to guide behavior after switching
5. **Error handling**: Use `safe_agent_call()` to catch and diagnose provider errors gracefully

**Next**: In `02_intermediate.ipynb`, we'll build multi-agent systems where different agents use different models simultaneously — enabling specialized pipelines and the agents-as-tools pattern.